# Efficient VLM Inference via Visual Token Compression

Colab demo for Qwen2.5-VL / Qwen2-VL inference-time visual token compression benchmarks.

## 1. Install dependencies

Use an A100 runtime when possible. Colab normally ships with a CUDA-matched PyTorch build, so the torch install line is kept as an optional fallback.

In [ ]:
try:
    import torch
except ImportError:
    %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124

%pip install -q -U torchvision transformers accelerate qwen-vl-utils datasets pillow pandas matplotlib tqdm psutil pynvml pyyaml

# If you see KeyError: 'qwen2_5_vl' or 'qwen2_vl', run this and restart runtime:
# %pip install -q -U git+https://github.com/huggingface/transformers accelerate

## 2. Clone repo and move to project root

This clones the public GitHub repo into Colab and adds the repo root to `sys.path`, so imports like `from src.compression import create_compression_method` work.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/YuqiWang26/vlm.git"
PROJECT_DIR = "/content/vlm"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=False)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Current directory:", os.getcwd())
print("Repo root added to sys.path:", PROJECT_DIR)

## 3. Check GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("Memory GB:", round(props.total_memory / 1024**3, 1))
!nvidia-smi

## 4. Smoke test compression functions without loading the VLM

In [ ]:
import torch
from src.compression import create_compression_method

device = "cuda" if torch.cuda.is_available() else "cpu"
tokens = torch.randn(1, 64, 128, device=device)
for name in ["none", "fixed", "importance", "merging"]:
    method = create_compression_method(name, retention_ratio=0.5, apply_proxy_image_budget=False)
    compressed = method.compress_visual_tokens(tokens).tokens
    print(name, tuple(tokens.shape), "->", tuple(compressed.shape))

## 5. Quick benchmark

This loads the model and runs a tiny benchmark on synthetic images. On A100, start with Qwen2.5-VL-3B. If model download or memory is a problem, use the Qwen2-VL-2B fallback cell.

In [ ]:
!python run_benchmark.py --quick --model-id Qwen/Qwen2.5-VL-3B-Instruct --dtype fp16 --attn-implementation eager --methods none,fixed,importance,merging --ratios 1.0,0.75,0.5,0.25 --no-plots

In [ ]:
# Fallback option:
# !python run_benchmark.py --quick --model-id Qwen/Qwen2-VL-2B-Instruct --dtype fp16 --attn-implementation eager --methods none,fixed,importance,merging --ratios 1.0,0.75,0.5,0.25 --no-plots

## 6. Full benchmark

In [ ]:
# This can take a while. Reduce --samples or remove high/4-image settings if needed.
# !python run_benchmark.py \
#   --model-id Qwen/Qwen2.5-VL-3B-Instruct \
#   --dtype fp16 \
#   --attn-implementation eager \
#   --methods none,fixed,importance,merging \
#   --ratios 1.0,0.75,0.5,0.25,0.1 \
#   --resolutions low,medium,high \
#   --num-images 1,2,4 \
#   --samples 3 \
#   --max-new-tokens 64 \
#   --no-plots

## 7. View results and plots

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

raw_path = "results/benchmark_results.csv"
summary_path = "results/summary_results.csv"

if not os.path.exists(raw_path):
    raise FileNotFoundError("Run the quick benchmark cell before viewing results.")

raw_df = pd.read_csv(raw_path)
display(raw_df.head())

if os.path.exists(summary_path):
    display(pd.read_csv(summary_path))

if "success" in raw_df.columns and not raw_df["success"].astype(bool).any():
    print("No successful benchmark rows. First errors:")
    error_cols = [col for col in ["compression_method", "retention_ratio", "oom", "error"] if col in raw_df.columns]
    display(raw_df[error_cols].head(10))

success_df = raw_df[raw_df["success"].astype(bool)].copy() if "success" in raw_df.columns else raw_df.copy()
if success_df.empty:
    raise RuntimeError("No successful benchmark rows to plot. Check the error column above.")

grouped = (
    success_df.groupby(["compression_method", "retention_ratio"], as_index=False)
    .agg(
        latency_ms=("latency_ms", "mean"),
        memory_mb=("peak_gpu_memory_mb", "mean"),
        quality_score=("quality_score", "mean"),
        throughput=("throughput_tokens_per_second", "mean"),
    )
)
display(grouped.sort_values(["compression_method", "retention_ratio"]))

baseline_rows = grouped[grouped["compression_method"] == "none"]
baseline = baseline_rows.iloc[0] if len(baseline_rows) else None
plot_df = grouped[grouped["compression_method"] != "none"].copy()

if baseline is not None:
    plot_df["latency_reduction_pct"] = (baseline["latency_ms"] - plot_df["latency_ms"]) / baseline["latency_ms"] * 100
    plot_df["memory_reduction_pct"] = (baseline["memory_mb"] - plot_df["memory_mb"]) / baseline["memory_mb"] * 100
else:
    plot_df["latency_reduction_pct"] = np.nan
    plot_df["memory_reduction_pct"] = np.nan

colors = {"fixed": "#0072B2", "importance": "#D55E00", "merging": "#009E73"}
labels = {"fixed": "Fixed pruning (internal)", "importance": "Importance pruning", "merging": "Token merging"}

fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=150)
axes = axes.ravel()

def style_axis(ax, title, ylabel):
    ax.set_title(title, fontsize=15, pad=10)
    ax.set_xlabel("Retention ratio", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, alpha=0.28)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

def plot_metric(ax, y_col, title, ylabel, baseline_col=None, percent=False):
    for method in ["fixed", "importance", "merging"]:
        sub = plot_df[plot_df["compression_method"] == method].sort_values("retention_ratio")
        if sub.empty:
            continue
        ax.plot(
            sub["retention_ratio"], sub[y_col],
            marker="o", linewidth=2.5, markersize=7,
            color=colors[method], label=labels[method],
        )
        for _, row in sub.iterrows():
            value = row[y_col]
            text = f"{value:.1f}%" if percent else f"{value:.0f}"
            ax.annotate(text, (row["retention_ratio"], value), xytext=(5, 5), textcoords="offset points", fontsize=8)
    if baseline is not None and baseline_col is not None:
        ax.axhline(baseline[baseline_col], color="#555555", linestyle="--", linewidth=1.7, label="No compression baseline")
    style_axis(ax, title, ylabel)

plot_metric(axes[0], "latency_ms", "Latency", "ms", baseline_col="latency_ms")
plot_metric(axes[1], "latency_reduction_pct", "Latency reduction vs baseline", "%", percent=True)
plot_metric(axes[2], "memory_reduction_pct", "Peak memory reduction vs baseline", "%", percent=True)
plot_metric(axes[3], "quality_score", "Answer quality", "keyword-match score", baseline_col="quality_score")

handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="upper center", ncol=4, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Visual token compression: efficiency-quality trade-off", fontsize=18, y=1.07)
fig.tight_layout()

os.makedirs("results", exist_ok=True)
plot_path = "results/notebook_tradeoff_summary.png"
fig.savefig(plot_path, bbox_inches="tight")
plt.show()
print(f"Saved notebook plot to {plot_path}")